In [1]:
!pip install -q \
    transformers==4.46.0 \
    huggingface_hub==0.26.0 \
    tokenizers==0.20.0 \
    jiwer \
    soundfile \
    librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.4/447.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.0 which is incompatible.


In [2]:
# 1. Import
import pandas as pd
import numpy as np
import torch
import librosa
import soundfile as sf
import os
from tqdm import tqdm
from jiwer import wer, cer
from transformers import Wav2Vec2Processor, Wav2Vec2ConformerForCTC

2026-03-03 15:42:26.279282: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772552546.481484      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772552546.558021      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772552547.021660      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772552547.021706      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772552547.021709      24 computation_placer.cc:177] computation placer alr

In [3]:
# 2. Config
CSV_PATH   = "/kaggle/input/datasets/eddiehoogewerf/mozilla-commonvoice/commonvoice-v24_en-AU/commonvoice-v24_en-AU.csv"
CLIPS_DIR  = "/kaggle/input/datasets/eddiehoogewerf/mozilla-commonvoice/commonvoice-v24_en-AU/audio_files"
AUDIO_DIR  = "/kaggle/working/cv_audio_wav"
MODEL_NAME = "facebook/wav2vec2-conformer-rope-large-960h-ft"  # Conformer pretrained on LibriSpeech 960h
os.makedirs(AUDIO_DIR, exist_ok=True)

In [4]:
# 3. Load CSV 
df = pd.read_csv(CSV_PATH)
df = df[["path", "sentence"]].dropna().reset_index(drop=True)
print(f"Total samples: {len(df)}")

Total samples: 55673


In [5]:
# 4. Convert MP3 → WAV 16kHz 
def convert_to_wav(mp3_path, wav_path, target_sr=16000):
    waveform, sr = librosa.load(mp3_path, sr=target_sr, mono=True)
    sf.write(wav_path, waveform, target_sr)

print("\nConverting MP3 → WAV 16kHz ...")
audio_paths = []
references  = []
skipped     = 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    mp3_path = os.path.join(CLIPS_DIR, row["path"])

    if not os.path.exists(mp3_path):
        skipped += 1
        continue

    wav_name = row["path"].replace(".mp3", ".wav")
    wav_path = os.path.join(AUDIO_DIR, wav_name)

    if not os.path.exists(wav_path):
        convert_to_wav(mp3_path, wav_path)

    audio_paths.append(wav_path)
    references.append(row["sentence"].upper().strip())

print(f"Ready: {len(audio_paths)} files | Skipped (missing): {skipped}")


Converting MP3 → WAV 16kHz ...


100%|██████████| 55673/55673 [39:21<00:00, 23.58it/s]

Ready: 55673 files | Skipped (missing): 0


In [6]:
# 5. Load Conformer Model
print(f"\nLoading model: {MODEL_NAME} ...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model     = Wav2Vec2ConformerForCTC.from_pretrained(MODEL_NAME)
model.eval()
model     = model.cuda()
print("Model loaded!")


Loading model: facebook/wav2vec2-conformer-rope-large-960h-ft ...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.37G [00:00<?, ?B/s]

Model loaded!


In [7]:
# 6. Inference
BATCH_SIZE = 16
hypotheses = []

print("\nRunning inference ...")
for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch_paths = audio_paths[i : i + BATCH_SIZE]

    # Load and pad audio
    batch_audio = [librosa.load(p, sr=16000, mono=True)[0] for p in batch_paths]

    inputs = processor(
        batch_audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values   = inputs.input_values.cuda()
    attention_mask = inputs.attention_mask.cuda()

    with torch.no_grad():
        logits = model(input_values, attention_mask=attention_mask).logits

    predicted_ids   = torch.argmax(logits, dim=-1)
    transcriptions  = processor.batch_decode(predicted_ids)
    hypotheses.extend([t.upper().strip() for t in transcriptions])


Running inference ...


100%|██████████| 3480/3480 [1:19:51<00:00,  1.38s/it]


In [8]:
# 7. Save Results
import re

def normalize(text):
    text = text.upper().strip()
    text = re.sub(r'[^\w\s]', '', text)  # remove all punctuation
    text = re.sub(r'\s+', ' ', text)     # collapse extra spaces
    return text

# Apply normalization to both lists
references = [normalize(r) for r in references]
hypotheses = [normalize(h) for h in hypotheses]

# Compute WER/CER on clean text
final_wer = wer(references, hypotheses)
final_cer = cer(references, hypotheses)

results_df = pd.DataFrame({
    "reference":  references,
    "hypothesis": hypotheses,
    "wer_sample": [wer([r], [h]) for r, h in zip(references, hypotheses)],
    "cer_sample": [cer([r], [h]) for r, h in zip(references, hypotheses)],
})
results_df.to_csv("/kaggle/working/conformer_results.csv", index=False)

In [9]:
# 8. Results
print(f"\n{'='*80}")
print(f"  SAMPLE PREDICTIONS")
print(f"{'='*80}")

for i, row in results_df.head(20).iterrows():
    print(f"\n[Sample {i+1}]")
    print(f"  REFERENCE  : {row['reference']}")
    print(f"  PREDICTION  : {row['hypothesis']}")
    print(f"  WER  : {row['wer_sample']*100:.1f}%  |  CER : {row['cer_sample']*100:.1f}%")
    print(f"  {'EXACT MATCH' if row['wer_sample'] == 0.0 else 'MISMATCH'}")
    print(f"  {'-'*76}")

print(f"\n{'='*80}")
print(f"  OVERALL RESULTS")
print(f"{'='*80}")
print(f"  Model         : {MODEL_NAME}")
print(f"  Total Samples : {len(results_df)}")
print(f"  WER           : {final_wer * 100:.2f}%")
print(f"  CER           : {final_cer * 100:.2f}%")
print(f"  Exact Matches : {(results_df['wer_sample'] == 0.0).sum()} / {len(results_df)} ({(results_df['wer_sample'] == 0.0).mean()*100:.1f}%)")
print(f"{'='*80}")


  SAMPLE PREDICTIONS

[Sample 1]
  REFERENCE  : PRINCESS VILAS HERSELF ALSO CONTRIBUTED PERSONALLY TO THE CONSTRUCTION OF THE TEMPLE
  PREDICTION  : PRINCESS PHYLLIS HERSELF ALSO CONTRIBUTED PERSONALLY TO THE CONSTRUCTION OF THE TEMPLE
  WER  : 8.3%  |  CER : 6.0%
  MISMATCH
  ----------------------------------------------------------------------------

[Sample 2]
  REFERENCE  : HE HAS ALSO SERVED IN THE CHAMBER OF DEPUTIES
  PREDICTION  : HE HAS ALSO SERVED IN THE CHAMBER OF DEPUTIES
  WER  : 0.0%  |  CER : 0.0%
  EXACT MATCH
  ----------------------------------------------------------------------------

[Sample 3]
  REFERENCE  : MOST OF HIS SUBJECTS WERE FOUND IN DEVON AND CORNWALL
  PREDICTION  : MOST OF HIS SUBJECTS WERE FOUND IN DEVON AND CORNWALL
  WER  : 0.0%  |  CER : 0.0%
  EXACT MATCH
  ----------------------------------------------------------------------------

[Sample 4]
  REFERENCE  : SHOTS RANG OUT AS THEY FLED TOWARDS THE AUSTRIAN CAMP
  PREDICTION  : SHOTS RANG OUT AS